# Supplamental Cloud Cover Analysis

Author: Daniel Vandevort

Date: June 2026

contact: vandevod@oregonstate.edu

### Note regarding string names for input date files
As with most data science projects, input datafiles may have different names for input data (like the column names in a dataframe) than what may exist in your input datafiles. For names that are not generated within the code, ensure that your input data files are named to match or, perhaps more simply, change the string names in this code to match your input data. The naming convention itself has no bearing on the quality of analysis (but certainly on the readability...) and is subjective to the user. Do what makes sense to you :)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.ticker as mticker
import calendar
from scipy import stats

## Useable Images (<50% cloud cover)

In [ ]:
cloud_df = pd.read_csv(r'path/to/cloud_cover_data.csv')  # Update the path to your CSV file
print(len(cloud_df))

In [ ]:
THRESHOLD = 50
CLASS_COLORS = {
    'cloud_free':    '#2196F3',  # blue
    'partial_cloud': '#90CAF9',  # light blue
    'cloudy':        '#B0BEC5',  # grey
}
CLASS_LABELS = {
    'cloud_free':    'Cloud-free (0%)',
    'partial_cloud': f'Partial cloud (0–{THRESHOLD}%)',
    'cloudy':        f'Cloudy (≥{THRESHOLD}%)',
}
CLASS_ORDER = ['cloud_free', 'partial_cloud', 'cloudy']

# ── Monthly pivot tables ───────────────────────────────────────────────────────
monthly_counts = (
    cloud_df.groupby(['year_month', 'cloud_class'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CLASS_ORDER, fill_value=0)
    .sort_index()
)

monthly_pct = monthly_counts.div(monthly_counts.sum(axis=1), axis=0) * 100

# ── mean % per calendar month ─────────────────────────
cloud_df['month'] = pd.to_datetime(cloud_df['year_month']).dt.month
seasonal_counts = (
    cloud_df.groupby(['month', 'cloud_class'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CLASS_ORDER, fill_value=0)
)
seasonal_pct = seasonal_counts.div(seasonal_counts.sum(axis=1), axis=0) * 100
# Reorder months to reflect ice season: Oct → Jun
month_order = [10, 11, 12, 1, 2, 3, 4, 5, 6]
month_labels = ['Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
seasonal_pct = seasonal_pct.reindex(month_order)

# ── X-axis tick positions (one label per Oct = start of each winter) ──────────
x_labels = monthly_counts.index.tolist()
oct_ticks = [i for i, ym in enumerate(x_labels) if ym.endswith('-10')]

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 14))

# ── Plot 1: Stacked bar — raw counts ─────────────────────────────────────────
ax1 = axes[0]
bottom = np.zeros(len(monthly_counts))
for cls in CLASS_ORDER:
    vals = monthly_counts[cls].values
    ax1.bar(range(len(monthly_counts)), vals, bottom=bottom,
            color=CLASS_COLORS[cls], label=CLASS_LABELS[cls], width=0.85)
    bottom += vals
ax1.set_title('A', loc = 'right', fontsize=18)
ax1.set_ylabel('Number of Images', fontsize=18)
ax1.set_xticks(oct_ticks)
ax1.set_xticklabels([x_labels[i][:4] for i in oct_ticks], rotation=45, ha='right', fontsize=18)
ax1.legend(loc='upper left', fontsize=14)
ax1.set_xlim(-0.5, len(monthly_counts) - 0.5)
ax1.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# ── Plot 2: Proportional stacked area — percentages ───────────────────────────
ax2 = axes[1]
bottom = np.zeros(len(monthly_pct))
for cls in CLASS_ORDER:
    vals = monthly_pct[cls].values
    ax2.bar(range(len(monthly_pct)), vals, bottom=bottom,
            color=CLASS_COLORS[cls], label=CLASS_LABELS[cls], width=0.85)
    bottom += vals
ax2.set_title('B', loc = 'right', fontsize=18)
ax2.set_ylabel('Proportion (%)', fontsize=18)
ax2.set_ylim(0, 100)
ax2.set_xticks(oct_ticks)
ax2.set_xticklabels([x_labels[i][:4] for i in oct_ticks], rotation=45, ha='right', fontsize=18)
ax2.set_xlim(-0.5, len(monthly_pct) - 0.5)

# ── Plot 3: mean % per cal. month ──────────────────────────────────────────────
ax3 = axes[2]
bottom = np.zeros(len(seasonal_pct))
for cls in CLASS_ORDER:
    vals = seasonal_pct[cls].values
    ax3.bar(range(len(seasonal_pct)), vals, bottom=bottom,
            color=CLASS_COLORS[cls], label=CLASS_LABELS[cls], width=0.6)
    bottom += vals

ax3.set_title('C', loc = 'right', fontsize=18)
ax3.set_ylabel('Proportion (%)', fontsize=18)
ax3.set_ylim(0, 100)
ax3.set_xticks(range(len(month_labels)))
ax3.set_xticklabels(month_labels, fontsize=18)

plt.tight_layout()
# plt.savefig(r'path/to/output_figure.png', dpi=500, bbox_inches='tight') # update as needed
plt.show()

# All images (regardless of cloud cover)
This analysis looks at the full time period of HLS-2 imagery (2014–2024) and calculates the percentage of winter-season days (Oct–Jun) that have at least one HLS-2 image available, regardless of cloud cover. It also examines the distribution of gaps between consecutive images during two seasonal windows: early winter (Oct–Jan) and late winter (Apr–Jun). NOTE: it does this with respect to the whole AOI: The CMR; This does not account for differing coverage per lake per year, instead focusing on regional coverage for a coarse understanding of HLS-2 observation behavior

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
cloud2_df = pd.read_csv(r'path/to/data.csv')  # Update the path data file
cloud2_df['date'] = pd.to_datetime(cloud2_df['date'])
cloud2_df['year_month'] = cloud2_df['date'].dt.to_period('M')
useable_images_cloud2_df = cloud2_df[
    (cloud2_df["cloud_class"] != 'cloudy')  # get images used in study (Cloud coverage <50%)
]
daily = (
    useable_images_cloud2_df.groupby('date')
    .agg(n_tiles=('date', 'count'))   # number of HLS tiles acquired that day
    .reset_index()
    .sort_values('date')
)
daily['month'] = daily['date'].dt.month
daily['year']  = daily['date'].dt.year
imaged_dates = daily['date']  # one entry per day that has >= 1 tile

print(len(cloud2_df))
print(len(useable_images_cloud2_df)) # should be the number of usable images total
print(len(daily))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ANALYSIS 1: % of analysis days with at least one HLS-2 image, per year
# ─────────────────────────────────────────────────────────────────────────────

def get_analysis_days(year):
    """
    Return all calendar days in the Oct–Jun winter window for a given
    'winter year', defined as Oct(year-1) through Jun(year).
    e.g. year=2015 → 2014-10-01 to 2015-06-30
    """
    start = pd.Timestamp(f'{year - 1}-10-01')
    end   = pd.Timestamp(f'{year}-06-30')
    return pd.date_range(start, end, freq='D')

# Ice seasons: first full window starts Oct 2014 → Jun 2015 (year=2015)
# Last full window: Oct 2023 → Jun 2024 (year=2024)
ice_seasons = range(2015, 2025) # 2014 - 2024

records = []
for yr in ice_seasons:
    analysis_days = get_analysis_days(yr)
    n_total = len(analysis_days)
    # Days in this ice season that have at least one image
    imaged_days = daily[
        (daily['date'] >= analysis_days[0]) &
        (daily['date'] <= analysis_days[-1])
    ]
    n_imaged = len(imaged_days)
    pct = n_imaged / n_total * 100
    records.append({
        'ice_season': yr,
        'label': f'{yr-1}–{str(yr)[2:]}',  # e.g. "2014–15"
        'n_total_days': n_total,
        'n_imaged_days': n_imaged,
        'pct_covered': pct
    })

coverage_df = pd.DataFrame(records)

low_yr  = coverage_df.loc[coverage_df['pct_covered'].idxmin()]
high_yr = coverage_df.loc[coverage_df['pct_covered'].idxmax()]

print('=== Analysis 1: % of Ice Season Days with HLS-2 Coverage ===')
print(coverage_df[['label', 'n_total_days', 'n_imaged_days', 'pct_covered']].to_string(index=False))
print(f"\nRange: {low_yr['pct_covered']:.1f}% ({low_yr['label']}) "
      f"to {high_yr['pct_covered']:.1f}% ({high_yr['label']})")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ANALYSIS 2: CDFs of imagery gaps for two seasonal windows
# ─────────────────────────────────────────────────────────────────────────────

def compute_gaps(dates_series, months):
    """
    Given a sorted Series of dates, filter to the given months,
    then compute day-gaps between consecutive dates within each
    contiguous month-window block (gaps that cross window boundaries
    are excluded so we don't inflate gaps with off-season breaks).
    """
    filtered = dates_series[dates_series.dt.month.isin(months)].sort_values()
    gaps = []
    prev = None
    for curr in filtered:
        if prev is not None:
            gap = (curr - prev).days
            # Only count gaps that stay within the seasonal window
            # i.e. no month in between falls outside the window
            all_months_in_gap = {
                (prev + pd.Timedelta(days=d)).month
                for d in range(1, gap)
            }
            if all_months_in_gap.issubset(set(months)) or gap == 1:
                gaps.append(gap)
        prev = curr
    return np.array(gaps)

early_winter_months = [10, 11, 12, 1]   # Oct–Jan
late_winter_months  = [4, 5, 6]          # Apr–Jun

gaps_early = compute_gaps(imaged_dates, early_winter_months)
gaps_late  = compute_gaps(imaged_dates, late_winter_months)

def make_cdf(gaps):
    """ Return one (x, y) point per unique gap value, where y is the
    cumulative proportion of gaps <= x."""
    unique_gaps, counts = np.unique(gaps, return_counts=True)
    cumulative = np.cumsum(counts) / len(gaps)
    return unique_gaps, cumulative * 100

early_x, early_cdf = make_cdf(gaps_early)
late_x,  late_cdf  = make_cdf(gaps_late)

print('\n=== Analysis 2: Gap Statistics ===')
for label, gaps in [('Oct–Jan', gaps_early), ('Apr–Jun', gaps_late)]:
    print(f"\n{label}:")
    print(f"  N gaps:  {len(gaps)}")
    print(f"  Median:  {np.median(gaps):.1f} days")
    print(f"  Mean:    {np.mean(gaps):.1f} days")
    print(f"  90th %:  {np.percentile(gaps, 90):.1f} days")
    print(f"  Max:     {np.max(gaps):.1f} days")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PLOTTING
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(6, 8))
plt.subplots_adjust(hspace=0.45)  # increase vertical space between plots
# ── Plot 1: % coverage per winter year ───────────────────────────────────────
ax1 = axes[0]

bars = ax1.bar(coverage_df['label'], coverage_df['pct_covered'],
               color='k', width=0.7, edgecolor='white')

# Annotate each bar with its % value
for bar, pct in zip(bars, coverage_df['pct_covered']):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.5,
             f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
# '% of Winter Days (Oct–Jun) with ≥1 HLS-2 Image'
ax1.set_title('A', loc = 'right', fontsize=18)
ax1.set_ylabel('% of Days with Coverage', fontsize=16)
ax1.set_ylim(0, 100)
ax1.set_xlabel('Ice Season', fontsize=16)
ax1.tick_params(axis='x', rotation=45)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/100:.1f}'))

# ── Plot 2: CDFs of imagery gaps ──────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(early_x, early_cdf / 100 , color='#1565C0', linewidth=2, label='Oct–Jan')
ax2.plot(late_x,  late_cdf  / 100, color='#F57C00', linewidth=2, label='Apr–Jun')
# CDF of Imagery Gaps by Seasonal Window'
ax2.set_title('B', loc = 'right', fontsize=18)
ax2.set_xlabel('Gap Between Consecutive Images (days)', fontsize=16)
ax2.set_ylabel('Cumulative Probability', fontsize=16)
ax2.set_xlim(left=1)
ax2.set_ylim(0.5, 1.0)
# Major ticks every 0.2, labelled with one decimal place
ax2.yaxis.set_major_locator(mticker.MultipleLocator(0.1))
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}'))
# Minor ticks every 0.05, no labels
ax2.minorticks_on()  # must be called before set_minor_locator to ensure they render
ax2.yaxis.set_minor_locator(mticker.MultipleLocator(0.05))
ax2.xaxis.set_minor_locator(mticker.NullLocator())  # keep x axis minor tick-free
ax2.tick_params(which='major', axis = 'both', direction='inout', length=8,  top=True, right=True)
ax2.tick_params(which='minor', axis = 'y', direction='in',   length=4,  top=True, right=True)
ax2.grid(False)
ax2.legend(loc='lower right')

# plt.savefig(r'path/to/output/figure', dpi=500, bbox_inches='tight') # update as needed
